# 00 — Setup on Google Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balk21/stark-cls-finetune/blob/main/notebooks/00_setup_colab.ipynb)

Use this notebook on **Google Colab**. (On a Vast.ai server or your own computer use `00_setup.ipynb`.)

Colab gives a fresh machine in every session, so the environment, the dataset and the checkpoints are kept on
your **Google Drive** and restored at the start of each session. The exact same `vot1` environment as on
Vast.ai / a local machine is used (same package versions). Results are only identical on the same GPU type
(see "Results across GPUs" in the README).

**Before you start:** *Runtime → Change runtime type → GPU* (T4, L4 or A100).

| | First session (once) | Later sessions |
|---|---|---|
| `vot1` environment | created with micromamba (~5–10 min), cached on Drive | restored from Drive (~1–3 min) |
| Checkpoint (ST101, 180 MB) | downloaded to Drive | copied from Drive (seconds) |
| VOT-LT2020 dataset (17 GB) | downloaded (20–60 min), cached on Drive (10–20 min) | restored from Drive (a few min) |

**Google Drive space needed:** ≈ 26 GB (environment 7.8 GB + dataset 17 GB + checkpoint) plus the experiment outputs.
Everything is stored in the Drive folder `DRIVE_ROOT` set in the first cell (default `MyDrive/LOKAP/`; use the same
value in every notebook):

| Drive folder | Content |
|---|---|
| `cache/` | `vot1_env_<hash>.tar`, `votlt2020_sequences.tar` |
| `checkpoints/` | checkpoints (same layout as `checkpoints/` in the repository) |
| `outputs/` | experiment outputs; they survive the session, so an interrupted run can be resumed |

The other notebooks (`01_run_experiment.ipynb`, `02_compare.ipynb`) run the same preparation in their first cell,
so after this notebook has been run once, you can open them directly in Colab.

## 1. Prepare the session

Mounts Google Drive (a permission dialog appears), clones the repository and restores or creates the
environment, the checkpoint and the dataset. In the first session this takes a long time; later sessions take
a few minutes.

In [ ]:
# Run this cell once at the top of every notebook. On Google Colab it also prepares the session
# (clones the repository, restores the environment / dataset / checkpoints from Google Drive).
DRIVE_ROOT = "/content/drive/MyDrive/LOKAP"   # Colab only: Google Drive folder for everything (same in all notebooks)

import sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    REPO = Path("/content/stark-cls-finetune")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/balk21/stark-cls-finetune.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only"])   # get the latest fixes
    sys.path.insert(0, str(REPO / "notebooks"))
    import importlib, nbhelper as nb
    nb = importlib.reload(nb)   # use the code just pulled, also when this cell is run again
    nb.colab_bootstrap(drive_root=DRIVE_ROOT)
else:
    sys.path.insert(0, str(Path.cwd()))   # the notebooks/ folder (nbhelper.py lives here)
    import nbhelper as nb

## 2. Environment check

In [ ]:
nb.cli("check", check=False)

## 3. Smoke test

Loads the model and tracks the first 50 frames of a sequence (mean IoU > 0.5 means the setup is correct).

In [ ]:
nb.cli("smoke", "--frames", "50", "--set", "ft_mode=online")

Setup complete. Open **`01_run_experiment.ipynb`** in Colab:
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balk21/stark-cls-finetune/blob/main/notebooks/01_run_experiment.ipynb)

---
## Optional

**More official checkpoints** (e.g. STARK-ST50): run the cell below with the models you need.

**Your own checkpoint** (e.g. `STARKSTcoco_ep0050.pth.tar`): upload it to
`MyDrive/LOKAP/checkpoints/stark_st2/baseline_R101/`; it is copied to the session automatically.
Then set `checkpoint="STARKSTcoco_ep0050.pth.tar"` in the experiment.

**Rebuild the environment** (only needed if `environment/vot1_environment.yml` changes; a changed file is detected
automatically): `nb.colab_bootstrap(drive_root=DRIVE_ROOT, rebuild_env=True)`.

In [ ]:
# nb.colab_bootstrap(drive_root=DRIVE_ROOT, checkpoints=[("stark_st", "baseline_R101"), ("stark_st", "baseline")])